In [ ]:
#############################
## ÚLTIMA VERSÃO 13052026
#############################


!pip install pandas numpy matplotlib seaborn scipy statsmodels scikit-learn openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import glob
import plotly.express as px
import requests
import re
import json
import csv
import os
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Craindo diretórios
d1 = "/content/drive/My Drive/PNAD/CAGED/SP"
d2 = "/content/drive/My Drive/PNAD/CAGED/PORTO"
d3 = "/content/drive/My Drive/PNAD/CAGED/PORTO_OCUPADOS"
d4 = "/content/drive/My Drive/PNAD/CAGED/SIMESPI"
d5 = "/content/drive/My Drive/PNAD/CAGED/SIMESPI_OCUPADOS"
d6 = "/content/drive/My Drive/PNAD/CAGED/SP_LACTO"
d7 = "/content/drive/My Drive/PNAD/CAGED/PORTO_LACTO"


# Função auxiliar para ler todos os arquivos de um padrão
def ler_arquivos(diretorio, padrao):
    arquivos = glob.glob(os.path.join(diretorio, padrao))
    lista_df = [pd.read_excel(arq) for arq in arquivos]
    return pd.concat(lista_df, ignore_index=True)

# Lendo e unificando os arquivos
dados_MEDIA_SP            = ler_arquivos(d1, "*_media.xlsx")
dados_ADMITIDOS_SP        = ler_arquivos(d1, "*_NUMADMITIDO.xlsx")
dados_MEDIA_SP_LACTO      = ler_arquivos(d6, "*_media.xlsx")
dados_ADMITIDOS_SP_LACTO  = ler_arquivos(d6, "*_NUMADMITIDO.xlsx")
dados_MEDIA_PORTO         = ler_arquivos(d2, "*_media.xlsx")
dados_ADMITIDOS_PORTO     = ler_arquivos(d2, "*_NUMADMITIDO.xlsx")
dados_MEDIA_PORTO_OCUPADOS   = ler_arquivos(d3, "*_media.xlsx")
dados_ADMITIDOS_PORTO_OCUPADOS = ler_arquivos(d3, "*_NUMADMITIDO.xlsx")
dados_MEDIA_PORTO_LACTO      = ler_arquivos(d7, "*_media.xlsx")
dados_ADMITIDOS_PORTO_LACTO  = ler_arquivos(d7, "*_NUMADMITIDO.xlsx")
dados_MEDIA_SIMESPI       = ler_arquivos(d4, "*_media.xlsx")
dados_ADMITIDOS_SIMESPI   = ler_arquivos(d4, "*_NUMADMITIDO.xlsx")
dados_MEDIA_SIMESPI_OCUPADOS  = ler_arquivos(d5, "*_media.xlsx")
dados_ADMITIDOS_SIMESPI_OCUPADOS = ler_arquivos(d5, "*_NUMADMITIDO.xlsx")

In [ ]:
#### PREPARANDO INPC PARA AGREGAR A BASE DE MÉDIAS

INPC = pd.read_csv("/content/drive/My Drive/PNAD/CAGED/indices.csv")

# Transformar de wide para long (Jan-Dez viram uma coluna 'mes')
inpc = INPC.melt(
    id_vars=["Ano"],
    value_vars=["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"],
    var_name="mes",
    value_name="inpc")

# Renomear categorias de mês para números
mapa_meses = {
    "Jan":"01","Fev":"02","Mar":"03","Abr":"04","Mai":"05","Jun":"06",
    "Jul":"07","Ago":"08","Set":"09","Out":"10","Nov":"11","Dez":"12"}
inpc["mes"] = inpc["mes"].map(mapa_meses)

# 5. Concatenar ano+mes
inpc["Anomes"] = inpc["Ano"].astype(str) + inpc["mes"]

# 6. Selecionar apenas colunas desejadas
inpc = inpc[["Anomes", "inpc"]]

In [ ]:
########################################################
#### AGREGANDO INPC A BASE DE MÉDIAS
########################################################

# Renomeando colunas
dados_MEDIA_SP = dados_MEDIA_SP.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "X": "Media_Salario" })

dados_MEDIA_SP_LACTO = dados_MEDIA_SP_LACTO.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "X": "Media_Salario" })

dados_MEDIA_PORTO = dados_MEDIA_PORTO.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "X": "Media_Salario" })

dados_MEDIA_PORTO_OCUPADOS = dados_MEDIA_PORTO_OCUPADOS.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "X": "Media_Salario" })
dados_MEDIA_PORTO_OCUPADOS_LACTO = dados_MEDIA_PORTO_OCUPADOS.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "X": "Media_Salario" })

dados_MEDIA_SIMESPI = dados_MEDIA_SIMESPI.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "X": "Media_Salario" })

dados_MEDIA_SIMESPI_OCUPADOS = dados_MEDIA_SIMESPI_OCUPADOS.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "X": "Media_Salario" })

# Ajustando categorias da coluna 'contrato'
mapa_contrato = {-1: "Demitidos", 1: "Admitidos"}

dados_MEDIA_SP["contrato"] = dados_MEDIA_SP["contrato"].map(mapa_contrato)
dados_MEDIA_SP_LACTO["contrato"] = dados_MEDIA_SP_LACTO["contrato"].map(mapa_contrato)
dados_MEDIA_PORTO["contrato"] = dados_MEDIA_PORTO["contrato"].map(mapa_contrato)
dados_MEDIA_PORTO_OCUPADOS["contrato"] = dados_MEDIA_PORTO_OCUPADOS["contrato"].map(mapa_contrato)
dados_MEDIA_PORTO_LACTO["contrato"] = dados_MEDIA_PORTO_LACTO["contrato"].map(mapa_contrato)
dados_MEDIA_SIMESPI["contrato"] = dados_MEDIA_SIMESPI["contrato"].map(mapa_contrato)
dados_MEDIA_SIMESPI_OCUPADOS["contrato"] = dados_MEDIA_SIMESPI_OCUPADOS["contrato"].map(mapa_contrato)

In [ ]:
###############################################################
#### Organizando os bancos de Número de admitidos para plotagem
###############################################################

# Renomeando colunas
dados_ADMITIDOS_SP = dados_ADMITIDOS_SP.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "n": "n" })

dados_ADMITIDOS_SP_LACTO = dados_ADMITIDOS_SP_LACTO.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "n": "n" })

dados_ADMITIDOS_PORTO = dados_ADMITIDOS_PORTO.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "n": "n"  })

dados_ADMITIDOS_PORTO_OCUPADOS = dados_ADMITIDOS_PORTO_OCUPADOS.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "n": "n" })
dados_ADMITIDOS_PORTO_LACTO = dados_ADMITIDOS_PORTO_LACTO.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "n": "n" })

dados_ADMITIDOS_SIMESPI = dados_ADMITIDOS_SIMESPI.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "n": "n"  })

dados_ADMITIDOS_SIMESPI_OCUPADOS = dados_ADMITIDOS_SIMESPI_OCUPADOS.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "n": "n" })


# Ajustando categorias da coluna 'contrato'
mapa_contrato = {-1: "Demitidos", 1: "Admitidos"}

dados_ADMITIDOS_SP["contrato"] = dados_ADMITIDOS_SP["contrato"].map(mapa_contrato)
dados_ADMITIDOS_SP_LACTO["contrato"] = dados_ADMITIDOS_SP_LACTO["contrato"].map(mapa_contrato)
dados_ADMITIDOS_PORTO["contrato"] = dados_ADMITIDOS_PORTO["contrato"].map(mapa_contrato)
dados_ADMITIDOS_PORTO_OCUPADOS["contrato"] = dados_ADMITIDOS_PORTO_OCUPADOS["contrato"].map(mapa_contrato)
dados_ADMITIDOS_PORTO_LACTO["contrato"] = dados_ADMITIDOS_PORTO_LACTO["contrato"].map(mapa_contrato)
dados_ADMITIDOS_SIMESPI["contrato"] = dados_ADMITIDOS_SIMESPI["contrato"].map(mapa_contrato)
dados_ADMITIDOS_SIMESPI_OCUPADOS["contrato"] = dados_ADMITIDOS_SIMESPI_OCUPADOS["contrato"].map(mapa_contrato)

In [ ]:
### AGREGANDO INPC NAS BASES DE MÉDIAS por ano/mes
################################################################################

# Garantir que 'Anomes' seja string em todas as bases
dados_MEDIA_SP["Anomes"] = dados_MEDIA_SP["Anomes"].astype(str)
dados_MEDIA_SP_LACTO["Anomes"] = dados_MEDIA_SP_LACTO["Anomes"].astype(str)
dados_MEDIA_PORTO["Anomes"] = dados_MEDIA_PORTO["Anomes"].astype(str)
dados_MEDIA_PORTO_OCUPADOS["Anomes"] = dados_MEDIA_PORTO_OCUPADOS["Anomes"].astype(str)
dados_MEDIA_PORTO_LACTO["Anomes"] = dados_MEDIA_PORTO_LACTO["Anomes"].astype(str)
dados_MEDIA_SIMESPI["Anomes"] = dados_MEDIA_SIMESPI["Anomes"].astype(str)
dados_MEDIA_SIMESPI_OCUPADOS["Anomes"] = dados_MEDIA_SIMESPI_OCUPADOS["Anomes"].astype(str)

inpc["Anomes"] = inpc["Anomes"].astype(str)

# Merge com INPC
dados_MEDIA_SP = pd.merge(dados_MEDIA_SP, inpc, on="Anomes")
dados_MEDIA_SP_LACTO = pd.merge(dados_MEDIA_SP_LACTO, inpc, on="Anomes")
dados_MEDIA_PORTO = pd.merge(dados_MEDIA_PORTO, inpc, on="Anomes")
dados_MEDIA_PORTO_OCUPADOS = pd.merge(dados_MEDIA_PORTO_OCUPADOS, inpc, on="Anomes")
dados_MEDIA_PORTO_LACTO = pd.merge(dados_MEDIA_PORTO_LACTO, inpc, on="Anomes")
dados_MEDIA_SIMESPI = pd.merge(dados_MEDIA_SIMESPI, inpc, on="Anomes")
dados_MEDIA_SIMESPI_OCUPADOS = pd.merge(dados_MEDIA_SIMESPI_OCUPADOS, inpc, on="Anomes")

# Função auxiliar para limpar INPC
def limpar_inpc(df):
    df["inpc"] = (
        df["inpc"]
        .astype(str)                # garante que é string
        .str.replace(",", ".", regex=False)  # troca vírgula por ponto
        .str.replace("%", "", regex=False)   # remove símbolo de porcentagem
    )
    df["inpc"] = pd.to_numeric(df["inpc"], errors="coerce")  # converte para numérico
    return df

# 3. Aplicar limpeza em cada base
dados_MEDIA_SP = limpar_inpc(dados_MEDIA_SP)
dados_MEDIA_SP_LACTO = limpar_inpc(dados_MEDIA_SP_LACTO)
dados_MEDIA_PORTO = limpar_inpc(dados_MEDIA_PORTO)
dados_MEDIA_PORTO_OCUPADOS = limpar_inpc(dados_MEDIA_PORTO_OCUPADOS)
dados_MEDIA_PORTO_LACTO = limpar_inpc(dados_MEDIA_PORTO_LACTO)
dados_MEDIA_SIMESPI = limpar_inpc(dados_MEDIA_SIMESPI)
dados_MEDIA_SIMESPI_OCUPADOS = limpar_inpc(dados_MEDIA_SIMESPI_OCUPADOS)

In [ ]:
#### AGREGANDO INPC ACUMULADO e DEFLACIONAMENTO
####  SP, Porto Feliz e Porto Feliz ocupados
#################################################

def calcular_inpc(df):
    # 1. Converter INPC em fator (1 + inpc/100)
    df["inpc_fator"] = 1 + df["inpc"] / 100

    # 2. Calcular INPC acumulado (produto cumulativo)
    df["inpc_acm"] = df["inpc_fator"].cumprod()

    # 3. Calcular salário deflacionado
    df["Sal_def_INPC"] = df["Media_Salario"] / df["inpc_acm"]

    return df

# Aplicar para cada base
dados_MEDIA_SP = calcular_inpc(dados_MEDIA_SP)
dados_MEDIA_SP_LACTO = calcular_inpc(dados_MEDIA_SP_LACTO)
dados_MEDIA_PORTO = calcular_inpc(dados_MEDIA_PORTO)
dados_MEDIA_PORTO_OCUPADOS = calcular_inpc(dados_MEDIA_PORTO_OCUPADOS)
dados_MEDIA_PORTO_LACTO = calcular_inpc(dados_MEDIA_PORTO_LACTO)
dados_MEDIA_SIMESPI = calcular_inpc(dados_MEDIA_SIMESPI)
dados_MEDIA_SIMESPI_OCUPADOS = calcular_inpc(dados_MEDIA_SIMESPI_OCUPADOS)

In [ ]:
###  GRÁFICO 1 - SALÁRIO MÉDIO DEFLACIONADO PELO INPC - SÃO PAULO

import plotly.express as px

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_MEDIA_SP_LACTO["Anomes"],
    "categoria": dados_MEDIA_SP_LACTO["contrato"],
    "inpc": dados_MEDIA_SP_LACTO["Sal_def_INPC"]
})

# Converter de long para wide (pivot)
SP1_inpc = temporario.pivot(index="time", columns="categoria", values="inpc")

# Converter coluna de tempo para formato de data
SP1_inpc.index = pd.to_datetime(SP1_inpc.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP1_inpc, x=SP1_inpc.index, y=SP1_inpc.columns,
              title="Salário médio deflacionado pelo INPC - São Paulo")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/SP_média.html", include_plotlyjs="cdn")

# Exibir
fig.show()

In [ ]:
###  GRÁFICO 11 - SALÁRIO MÉDIO DEFLACIONADO PELO INPC - SÃO PAULO_LACTO

import plotly.express as px

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_MEDIA_SP["Anomes"],
    "categoria": dados_MEDIA_SP["contrato"],
    "inpc": dados_MEDIA_SP["Sal_def_INPC"]
})

# Converter de long para wide (pivot)
SP11_inpc = temporario.pivot(index="time", columns="categoria", values="inpc")

# Converter coluna de tempo para formato de data
SP11_inpc.index = pd.to_datetime(SP11_inpc.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP11_inpc, x=SP11_inpc.index, y=SP11_inpc.columns,
              title="Salário médio deflacionado pelo INPC - São Paulo laticínios")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/SP_média.html", include_plotlyjs="cdn")

# Exibir
fig.show()

In [ ]:
###  GRÁFICO 2 - NÚMERO DE ADMITIDOS/DEMITIDOS - SÃO PAULO

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_ADMITIDOS_SP["Anomes"],
    "categoria": dados_ADMITIDOS_SP["contrato"],
    "n": dados_ADMITIDOS_SP["n"]
})

# Converter de long para wide
SP2_n = temporario.pivot(index="time", columns="categoria", values="n")

# Converter coluna de tempo para formato de data
SP2_n.index = pd.to_datetime(SP2_n.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP2_n, x=SP2_n.index, y=SP2_n.columns,
              title="Número de Admitidos e Demitidos - São Paulo")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/SP_Admitidos.html", include_plotlyjs="cdn")

fig.show()

In [ ]:
###  GRÁFICO 12 - NÚMERO DE ADMITIDOS/DEMITIDOS - SÃO PAULO_LACTO

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_ADMITIDOS_SP_LACTO["Anomes"],
    "categoria": dados_ADMITIDOS_SP_LACTO["contrato"],
    "n": dados_ADMITIDOS_SP_LACTO["n"]
})

# Converter de long para wide
SP12_n = temporario.pivot(index="time", columns="categoria", values="n")

# Converter coluna de tempo para formato de data
SP12_n.index = pd.to_datetime(SP12_n.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP12_n, x=SP12_n.index, y=SP12_n.columns,
              title="Número de Admitidos e Demitidos - São Paulo laticínios")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/SP_Admitidos.html", include_plotlyjs="cdn")

fig.show()

In [ ]:
###  GRÁFICO 3 - SALÁRIO MÉDIO DEFLACIONADO PELO INPC - PORTO FELIZ

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_MEDIA_PORTO["Anomes"],
    "categoria": dados_MEDIA_PORTO["contrato"],
    "inpc": dados_MEDIA_PORTO["Sal_def_INPC"]
})

# Converter de long para wide (pivot)
SP3_inpc = temporario.pivot(index="time", columns="categoria", values="inpc")

# Converter coluna de tempo para formato de data
SP3_inpc.index = pd.to_datetime(SP3_inpc.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP3_inpc, x=SP3_inpc.index, y=SP3_inpc.columns,
              title="Salário médio deflacionado pelo INPC - Porto Feliz (sem filtro de ocupação)")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/Porto_média.html", include_plotlyjs="cdn")

fig.show()

In [ ]:
###  GRÁFICO 4 - NÚMERO DE ADMITIDOS/DEMITIDOS - PORTO FELIZ

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_ADMITIDOS_PORTO["Anomes"],
    "categoria": dados_ADMITIDOS_PORTO["contrato"],
    "n": dados_ADMITIDOS_PORTO["n"]
})

# Converter de long para wide
SP4_n = temporario.pivot(index="time", columns="categoria", values="n")

# Converter coluna de tempo para formato de data
SP4_n.index = pd.to_datetime(SP4_n.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP4_n, x=SP4_n.index, y=SP4_n.columns,
              title="Número de Admitidos e Demitidos - Porto Feliz (sem filtro de ocupação)")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/Porto_Admitidos.html", include_plotlyjs="cdn")

fig.show()

In [ ]:
###  GRÁFICO 5 - SALÁRIO MÉDIO DEFLACIONADO PELO INPC - PORTO FELIZ OCUPADOS

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_MEDIA_PORTO_OCUPADOS["Anomes"],
    "categoria": dados_MEDIA_PORTO_OCUPADOS["contrato"],
    "inpc": dados_MEDIA_PORTO_OCUPADOS["Sal_def_INPC"]
})

# Converter de long para wide (pivot)
SP5_inpc = temporario.pivot(index="time", columns="categoria", values="inpc")

# Converter coluna de tempo para formato de data
SP5_inpc.index = pd.to_datetime(SP5_inpc.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP5_inpc, x=SP5_inpc.index, y=SP5_inpc.columns,
              title="Salário médio deflacionado pelo INPC - Porto Feliz (com filtro de ocupação)")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/Porto_Ocupados_média.html", include_plotlyjs="cdn")

fig.show()

In [ ]:
###  GRÁFICO 6 - NÚMERO DE ADMITIDOS/DEMITIDOS - PORTO FELIZ OCUPADOS

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_ADMITIDOS_PORTO_OCUPADOS["Anomes"],
    "categoria": dados_ADMITIDOS_PORTO_OCUPADOS["contrato"],
    "n": dados_ADMITIDOS_PORTO_OCUPADOS["n"]
})

# Converter de long para wide
SP6_n = temporario.pivot(index="time", columns="categoria", values="n")

# Converter coluna de tempo para formato de data
SP6_n.index = pd.to_datetime(SP6_n.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP6_n, x=SP6_n.index, y=SP6_n.columns,
              title="Número de Admitidos e Demitidos - Porto Feliz (com filtro de ocupação)")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/Porto_Ocupados_Admitidos.html", include_plotlyjs="cdn")

fig.show()

In [ ]:
###  GRÁFICO 13 - SALÁRIO MÉDIO DEFLACIONADO PELO INPC - PORTO FELIZ_LACTO

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_MEDIA_PORTO_LACTO["Anomes"],
    "categoria": dados_MEDIA_PORTO_LACTO["contrato"],
    "inpc": dados_MEDIA_PORTO_LACTO["Sal_def_INPC"]
})

# Converter de long para wide (pivot)
SP13_inpc = temporario.pivot(index="time", columns="categoria", values="inpc")

# Converter coluna de tempo para formato de data
SP13_inpc.index = pd.to_datetime(SP13_inpc.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP13_inpc, x=SP13_inpc.index, y=SP13_inpc.columns,
              title="Salário médio deflacionado pelo INPC - Porto Feliz (com filtro de laticínios)")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/Porto_Ocupados_média.html", include_plotlyjs="cdn")

fig.show()

In [ ]:
###  GRÁFICO 14 - NÚMERO DE ADMITIDOS/DEMITIDOS - PORTO FELIZ_LACTO

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_ADMITIDOS_PORTO_LACTO["Anomes"],
    "categoria": dados_ADMITIDOS_PORTO_LACTO["contrato"],
    "n": dados_ADMITIDOS_PORTO_LACTO["n"]
})

# Converter de long para wide
SP14_n = temporario.pivot(index="time", columns="categoria", values="n")

# Converter coluna de tempo para formato de data
SP14_n.index = pd.to_datetime(SP14_n.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP14_n, x=SP14_n.index, y=SP14_n.columns,
              title="Número de Admitidos e Demitidos - Porto Feliz (com filtro de laticínios)")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/Porto_Ocupados_Admitidos.html", include_plotlyjs="cdn")

fig.show()

In [ ]:
###  GRÁFICO 7 - SALÁRIO MÉDIO DEFLACIONADO PELO INPC - SIMESPI

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_MEDIA_SIMESPI["Anomes"],
    "categoria": dados_MEDIA_SIMESPI["contrato"],
    "inpc": dados_MEDIA_SIMESPI["Sal_def_INPC"]
})

# Converter de long para wide (pivot com agregação)
SP7_inpc = temporario.pivot_table(
    index="time",
    columns="categoria",
    values="inpc",
    aggfunc="mean"  # ou "sum", "max", dependendo do que você deseja
)

# Converter coluna de tempo para formato de data
SP7_inpc.index = pd.to_datetime(SP7_inpc.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(
    SP7_inpc,
    x=SP7_inpc.index,
    y=SP7_inpc.columns,
    title="Salário médio deflacionado pelo INPC - SIMESPI"
)

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/SIMESPI_média.html", include_plotlyjs="cdn")

fig.show()

In [ ]:
###  GRÁFICO 8 - NÚMERO DE ADMITIDOS/DEMITIDOS - SIMESPI

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_ADMITIDOS_SIMESPI["Anomes"],
    "categoria": dados_ADMITIDOS_SIMESPI["contrato"],
    "n": dados_ADMITIDOS_SIMESPI["n"]
})

# Converter de long para wide com agregação
SP8_n = temporario.pivot_table(
    index="time",
    columns="categoria",
    values="n",
    aggfunc="sum"
)

# Converter coluna de tempo para formato de data
SP8_n.index = pd.to_datetime(SP8_n.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(
    SP8_n,
    x=SP8_n.index,
    y=SP8_n.columns,
    title="Número de Admitidos e Demitidos - SIMESPI"
)

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/SIMESPI_Admitidos.html", include_plotlyjs="cdn")

fig.show()

In [ ]:
###  GRÁFICO 9 - SALÁRIO MÉDIO DEFLACIONADO PELO INPC - SIMESPI OCUPADOS

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_MEDIA_SIMESPI_OCUPADOS["Anomes"],
    "categoria": dados_MEDIA_SIMESPI_OCUPADOS["contrato"],
    "inpc": dados_MEDIA_SIMESPI_OCUPADOS["Sal_def_INPC"]
})

# Converter de long para wide (pivot com agregação)
SP9_inpc = temporario.pivot_table(
    index="time",
    columns="categoria",
    values="inpc",
    aggfunc="mean"  # ou "sum", "max", dependendo do que você deseja
)

# Converter coluna de tempo para formato de data
SP9_inpc.index = pd.to_datetime(SP9_inpc.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(
    SP9_inpc,
    x=SP9_inpc.index,
    y=SP9_inpc.columns,
    title="Salário médio deflacionado pelo INPC - SIMESPI OCUPADOS"
)

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/SIMESPI_Ocupados_média.html", include_plotlyjs="cdn")

fig.show()

In [ ]:
###  GRÁFICO 10 - NÚMERO DE ADMITIDOS/DEMITIDOS - SIMESPI OCUPADOS

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_ADMITIDOS_SIMESPI_OCUPADOS["Anomes"],
    "categoria": dados_ADMITIDOS_SIMESPI_OCUPADOS["contrato"],
    "n": dados_ADMITIDOS_SIMESPI_OCUPADOS["n"]
})

# Converter de long para wide com agregação
SP10_n = temporario.pivot_table(
    index="time",
    columns="categoria",
    values="n",
    aggfunc="sum"
)

# Converter coluna de tempo para formato de data
SP10_n.index = pd.to_datetime(SP10_n.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(
    SP10_n,
    x=SP10_n.index,
    y=SP10_n.columns,
    title="Número de Admitidos e Demitidos - SIMESPI OCUPADOS"
)

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/SIMESPI_OCUPADOS_Admitidos.html", include_plotlyjs="cdn")

fig.show()